In [ ]:
import os

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

os.chdir("..")
os.getcwd()

In [ ]:
aef_size = 256
unlabelled = True

DATA_DIR = os.environ.get("DATA_DIR", "data/")

aux_df = pd.read_csv(
    f'{DATA_DIR}/s2bms/model_ready_s2bms{"-unlabelled-merged" if unlabelled else ""}.csv'
)
aef_df = pd.read_csv(
    f'{DATA_DIR}/s2bms/eo/avr_aef_{aef_size}{"_unlabelled" if unlabelled else ""}.csv'
)

import torch  # only used here, to read the .pth split-index file; no tensors kept downstream

# split_indices = torch.load(
#     os.path.join(f'{DATA_DIR}/s2bms/splits/s2bms{"_unlabelled" if unlabelled else ""}_union_val_test.pth'),
#     weights_only=False,
# )
split_indices = torch.load(
    os.path.join(
        f"{DATA_DIR}/s2bms/splits/split_indices_s2bms+s2bms-unlabelled-20260529_2026-05-29-1438.pth"
    ),
    weights_only=False,
)
aux_idx = aux_df.name_loc
aef_idx = aef_df.name_loc

common_train_idx = pd.Series(
    list(set(split_indices["train_indices"]) & set(aef_idx) & set(aux_idx))
)
common_val_idx = pd.Series(list(set(split_indices["val_indices"]) & set(aef_idx) & set(aux_idx)))
common_test_idx = pd.Series(list(set(split_indices["test_indices"]) & set(aef_idx) & set(aux_idx)))
print(
    "train | val | test\n",
    len(common_train_idx),
    "|",
    len(common_val_idx),
    "|",
    len(common_test_idx),
)

In [ ]:
df_coords = pd.read_csv(f"{DATA_DIR}/s2bms/model_ready_s2bms-unlabelled-merged.csv")
gdf_coords = gpd.GeoDataFrame(
    df_coords, geometry=gpd.points_from_xy(df_coords["lon"], df_coords["lat"]), crs="EPSG:4326"
)
gdf_coords = gdf_coords.to_crs("EPSG:27700")
gdf_coords["x"] = gdf_coords.geometry.x
gdf_coords["y"] = gdf_coords.geometry.y

gdf_coords.plot("aux_bioclim_01")

In [ ]:
data_env = pd.read_csv("/Users/tplas/Downloads/gridxdata.csv")
data_pollution = pd.read_csv("/Users/tplas/Downloads/mappm252024g.csv")
ind_row_new_header = 4
new_columns = list(data_pollution.iloc[ind_row_new_header].values)
data_pollution = data_pollution[ind_row_new_header + 1 :]
data_pollution.columns = new_columns
for i_c, c in enumerate(data_pollution.columns):
    data_pollution[c] = data_pollution[c].replace("MISSING", np.nan)
    if c == "pm252024g":
        data_pollution[c] = data_pollution[c].astype(float)
    else:
        data_pollution[c] = data_pollution[c].astype(int)
data_pollution

In [ ]:
gdf_env = gpd.GeoDataFrame(
    data_env, geometry=gpd.points_from_xy(data_env["x"], data_env["y"]), crs="EPSG:27700"
)

# gdf_env = gdf_env.to_crs("EPSG:4326")
# gdf_env["lon"] = gdf_env.geometry.x
# gdf_env["lat"] = gdf_env.geometry.y

gdf_pollution = gpd.GeoDataFrame(
    data_pollution,
    geometry=gpd.points_from_xy(data_pollution["x"], data_pollution["y"]),
    crs="EPSG:27700",
)

# gdf_pollution = gdf_pollution.to_crs("EPSG:4326")
# gdf_pollution["lon"] = gdf_pollution.geometry.x
# gdf_pollution["lat"] = gdf_pollution.geometry.y

fig, ax = plt.subplots(1, 3, figsize=(10, 10))

gdf_coords.plot(ax=ax[0], column="aux_bioclim_01", markersize=1, cmap="terrain", legend=True)
gdf_env.plot(ax=ax[1], column="temp_avg", markersize=1, cmap="terrain", legend=True)
gdf_pollution.plot(ax=ax[2], column="pm252024g", markersize=1, cmap="terrain", legend=True)

In [ ]:
import numpy as np
from pyproj import Transformer
from scipy.spatial import cKDTree

query_x, query_y = gdf_coords.geometry.x.to_numpy(), gdf_coords.geometry.y.to_numpy()
gdf_final = gdf_coords.copy()
for prefix, gdf_ref in zip(["env", "pol"], [gdf_env, gdf_pollution]):  # reference grid

    ref_coords = gdf_ref[["x", "y"]].to_numpy()
    tree = cKDTree(ref_coords)

    # --- 3. query nearest reference point for every query point (single vectorized call) ---
    query_coords = np.column_stack([query_x, query_y])
    dist, idx = tree.query(query_coords, k=1)
    n_not_within_1km = sum(dist >= np.sqrt(500**2 + 500**2))
    print(f"Number of {prefix} points not within 1km: {n_not_within_1km}")
    # --- 4. attach matched reference rows + distance (metres) back onto the query gdf ---
    matched = gdf_ref.iloc[idx].reset_index(drop=True)
    gdf_final = gdf_final.join(matched.add_prefix(prefix + "_"))
    gdf_final[f"{prefix}_dist_nn_m"] = dist

In [ ]:
gdf_final[gdf_final["env_dist_nn_m"] != gdf_final["pol_dist_nn_m"]]